In [1]:
# Import tools
import csv
import io
import json
import re
import unicodedata
import zipfile
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from scipy.linalg import qr
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import haversine_distances
from sklearn.pipeline import Pipeline


In [2]:
# Set plot style
plt.rc("text",usetex=False)
plt.rc("font",family="serif")
plt.rc("mathtext",fontset="cm")
plt.rc("font",size=13)
plt.rc("axes",labelsize=14)
plt.rc("xtick",labelsize=11)
plt.rc("ytick",labelsize=11)
plt.rc("legend",fontsize=11)
plt.rc("xtick",top=False,direction="out")
plt.rc("ytick",right=False,direction="out")
plt.rc("xtick.major",size=4.5,width=0.9)
plt.rc("ytick.major",size=4.5,width=0.9)
plt.rc("lines",linewidth=1.8,markersize=4.5,markeredgewidth=0.8)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
colors = {"blue":"#0072B2","orange":"#D55E00","green":"#009E73","purple":"#CC79A7","gray":"#7F7F7F"}


In [3]:
# Set England protocol
START = 2018
END = 2022
VALIDATION = 2023
TEST = 2024
MONTHS = [6,7,8,9,10]
MIN_YEARS = 4
MIN_DAYS = 40
MIN_HOURS = 18
TARGET = "South East"
BUDGET = 10
RANDOM = 200
BOOTSTRAP = 1000
SEED = 7
AURN = "https://uk-air.defra.gov.uk"
IQAIR = "https://www.iqair.com/gb/wildfire/uk/england"
ACCESS = pd.Timestamp.now(tz="UTC").date().isoformat()
OUT = Path("england_results")

protocol = pd.DataFrame([
    ["qualification",f"{START}-{END}"],
    ["validation",VALIDATION],
    ["test",TEST],
    ["season","June-October"],
    ["minimum qualification years",MIN_YEARS],
    ["minimum days per year",MIN_DAYS],
    ["minimum valid hours per day",MIN_HOURS],
    ["working population target",TARGET],
    ["sensor budget",BUDGET],
    ["random sets",RANDOM],
    ["seed",SEED],
    ["access date UTC",ACCESS]
],columns=["parameter","value"])
display(protocol)


,parameter,value
0,qualification,2018-2022
1,validation,2023
2,test,2024
3,season,June-October
4,minimum qualification years,4
5,minimum days per year,40
6,minimum valid hours per day,18
7,working population target,South East
8,sensor budget,10
9,random sets,200


In [4]:
# Define AURN tools
def key(value):
    text = unicodedata.normalize("NFKD",str(value)).encode("ascii","ignore").decode()
    return re.sub(r"[^a-z0-9]","",text.lower())

def table(content):
    try:
        text = content.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = content.decode("cp1252")
    lines = text.splitlines()
    header = None
    for i,line in enumerate(lines):
        values = next(csv.reader([line]))
        if len(values)>=20 and any("date" in str(value).lower() for value in values):
            header = i
            break
    if header is None:
        raise ValueError("AURN header not found")
    data = pd.read_csv(io.StringIO("\n".join(lines[header:])),low_memory=False)
    dates = [column for column in data.columns if "date" in str(column).lower()]
    if len(dates)!=1:
        raise ValueError("Expected one date column")
    hours = [column for column in data.columns if column!=dates[0]]
    if len(hours)!=24:
        raise ValueError("Expected 24 hourly columns")
    return data,dates[0],hours


In [5]:
# Build AURN metadata
session = requests.Session()
session.headers.update({"User-Agent":"Mozilla/5.0"})
current = session.get(
    f"{AURN}/networks/find-sites?action=results&country_id=9999&group_id=4&location_type=9999&pollutant=&region_id=9999&site_name=&view=advanced",
    timeout=120
)
closed = session.get(f"{AURN}/networks/aurn-sites",timeout=120)
current.raise_for_status()
closed.raise_for_status()

currentids = sorted(set(re.findall(r"UKA\d{5}",current.text)))
page = BeautifulSoup(closed.text,"html.parser")
closedcodes = []
for link in page.find_all("a",href=True):
    match = re.search(r"(?:aurn-)?site-info\?site_id=([^&#\"']+)",link["href"])
    if match:
        closedcodes.append(match.group(1))
closedcodes = sorted(set(closedcodes))

refs = [("current",value,f"{AURN}/networks/site-info?provider=&uka_id={value}") for value in currentids]
refs += [("closed",value,f"{AURN}/networks/site-info?site_id={value}") for value in closedcodes]
rows = []
status = []
for i,(kind,site,url) in enumerate(refs,1):
    if i==1 or i%50==0:
        print("Reading AURN site",i,"of",len(refs))
    response = session.get(url,timeout=60)
    status.append({"kind":kind,"site":site,"status":response.status_code})
    if not response.ok:
        continue
    page = BeautifulSoup(response.text,"html.parser")
    text = page.get_text("\n",strip=True)
    uka = re.search(r"UK-AIR ID:\s*(UKA\d{5})",text)
    region = re.search(r"Government Region:\s*([^\n]+)",text)
    coords = re.search(r"Latitude/Longitude:\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)",text)
    code = None
    for link in page.find_all("a",href=True):
        match = re.search(r"flat_files\?site_id=([^&#\"']+)",link["href"])
        if match:
            code = match.group(1)
            break
    start = np.nan
    end = np.nan
    for tablerow in page.find_all("tr"):
        values = [cell.get_text(" ",strip=True) for cell in tablerow.find_all(["th","td"])]
        if not values:
            continue
        pollutant = key(values[0])
        if "pm25particulatematterhourlymeasured" in pollutant and "nonvolatile" not in pollutant and not pollutant.startswith("volatile"):
            if len(values)>=3:
                start,end = values[1],values[2]
            break
    rows.append({
        "uka":uka.group(1) if uka else np.nan,
        "code":code,
        "region":region.group(1).strip() if region else np.nan,
        "lat":float(coords.group(1)) if coords else np.nan,
        "lon":float(coords.group(2)) if coords else np.nan,
        "start":start,
        "end":end
    })

status = pd.DataFrame(status)
display(status["status"].value_counts().rename_axis("status").reset_index(name="pages"))
if not rows:
    raise RuntimeError("AURN site pages were not accessible")
meta = pd.DataFrame(rows)
meta["start"] = pd.to_datetime(meta["start"],errors="coerce",dayfirst=True)
meta["end"] = pd.to_datetime(meta["end"].replace({"-":np.nan,"":np.nan}),errors="coerce",dayfirst=True)
devolved = meta["region"].astype(str).str.contains("Scotland|Wales|Northern Ireland",case=False,regex=True,na=False)
meta = meta[
    ~devolved
    &meta["uka"].notna()
    &meta["code"].notna()
    &meta["lat"].notna()
    &meta["lon"].notna()
    &meta["start"].notna()
    &meta["start"].le(pd.Timestamp(f"{TEST}-10-31"))
    &(meta["end"].isna()|meta["end"].ge(pd.Timestamp(f"{START}-06-01")))
].drop_duplicates("uka").reset_index(drop=True)
print("Parsed site pages:",len(rows))
print("Current AURN IDs:",len(currentids))
print("Closed AURN references:",len(closedcodes))
print("Pre-freeze PM2.5 metadata sites:",len(meta))
display(meta[["uka","code","region","lat","lon","start","end"]].head())


Reading AURN site 1 of 325
Reading AURN site 50 of 325
Reading AURN site 100 of 325
Reading AURN site 150 of 325
Reading AURN site 200 of 325
Reading AURN site 250 of 325
Reading AURN site 300 of 325


,status,pages
0,200,325


Parsed site pages: 325
Current AURN IDs: 212
Closed AURN references: 113
Pre-freeze PM2.5 metadata sites: 91


,uka,code,region,lat,lon,start,end
0,UKA00012,SIB,Eastern,52.294400,1.463497,2024-01-04,NaT
1,UKA00152,LH,South East,50.793700,0.181250,2022-06-01,NaT
2,UKA00168,YW,South West,50.597600,-3.716510,2022-07-01,NaT
3,UKA00169,HM,Yorkshire & Humberside,54.334497,-0.808820,2022-05-25,NaT
4,UKA00170,GLAZ,North West & Merseyside,53.460080,-2.472056,2022-06-01,NaT


In [6]:
# Download England PM2.5
rows = []
logs = []
for i,row in enumerate(meta.itertuples(),1):
    if i==1 or i%20==0:
        print("Downloading site",i,"of",len(meta))
    for year in range(START,TEST+1):
        if row.start>pd.Timestamp(f"{year}-10-31"):
            continue
        if pd.notna(row.end) and row.end<pd.Timestamp(f"{year}-06-01"):
            continue
        url = f"{AURN}/datastore/data_files/site_pol_data/{row.code}_PM25_{year}.csv"
        response = session.get(url,timeout=60)
        kind = response.headers.get("content-type","").lower()
        if not response.ok or "html" in kind:
            logs.append({"site":row.uka,"year":year,"status":"missing"})
            continue
        try:
            data,datecol,hours = table(response.content)
            dates = pd.to_datetime(data[datecol],errors="coerce",dayfirst=True,format="mixed")
            values = data[hours].apply(pd.to_numeric,errors="coerce")
            part = pd.DataFrame({
                "site":row.uka,
                "date":dates,
                "hours":values.notna().sum(axis=1),
                "pm":values.mean(axis=1),
                "lat":row.lat,
                "lon":row.lon,
                "region":row.region
            })
            part = part[
                part["date"].dt.month.isin(MONTHS)
                &part["hours"].ge(MIN_HOURS)
                &part["pm"].notna()
            ].copy()
            part["year"] = year
            rows.append(part)
            logs.append({"site":row.uka,"year":year,"status":"ok","days":len(part)})
        except Exception as error:
            logs.append({"site":row.uka,"year":year,"status":"parse_error","detail":str(error)})
if not rows:
    raise RuntimeError("No England PM2.5 files parsed")
daily = pd.concat(rows,ignore_index=True)
logs = pd.DataFrame(logs)
print("Daily rows:",len(daily))
print("Sites with data:",daily["site"].nunique())
display(logs["status"].value_counts().rename_axis("status").reset_index(name="files"))


Daily rows: 61489
Sites with data: 87


,status,files
0,ok,452
1,missing,6


In [7]:
# Freeze England candidates
coverage = daily.groupby(["site","year"],as_index=False).agg(days=("date","nunique"))
traincoverage = coverage[coverage["year"].between(START,END)].copy()
stats = traincoverage.groupby("site",as_index=False).agg(
    years=("year","nunique"),
    days=("days","median"),
    total=("days","sum")
)
stats = stats.merge(
    meta[["uka","code","region","lat","lon"]],
    left_on="site",
    right_on="uka",
    how="left",
    validate="one_to_one"
).drop(columns="uka")
english = {
    "North East":"North East",
    "North West & Merseyside":"North West",
    "Yorkshire & Humberside":"Yorkshire and The Humber",
    "East Midlands":"East Midlands",
    "West Midlands":"West Midlands",
    "Eastern":"East of England",
    "Greater London":"London",
    "South East":"South East",
    "South West":"South West"
}
stats["onsregion"] = stats["region"].map(english)
stats["qualified"] = stats["years"].ge(MIN_YEARS)&stats["days"].ge(MIN_DAYS)&stats["onsregion"].notna()
sites = stats.loc[stats["qualified"],"site"].sort_values().tolist()
candidates = stats.loc[
    stats["qualified"],["site","code","onsregion","lat","lon"]
].rename(columns={"onsregion":"region"}).sort_values("site").reset_index(drop=True)
candidateaudit = pd.DataFrame({
    "metric":[
        "England candidates","South East candidates","2023 observed candidates",
        "2024 observed candidates","Oxford St Ebbes retained","Inverness excluded"
    ],
    "value":[
        len(sites),
        candidates["region"].eq(TARGET).sum(),
        daily[daily["site"].isin(sites)&daily["year"].eq(VALIDATION)]["site"].nunique(),
        daily[daily["site"].isin(sites)&daily["year"].eq(TEST)]["site"].nunique(),
        "UKA00518" in sites,
        "UKA00434" not in sites
    ]
})
regions = candidates["region"].value_counts().rename_axis("region").reset_index(name="sites")
display(candidateaudit)
display(regions)


,metric,value
0,England candidates,52
1,South East candidates,9
2,2023 observed candidates,51
3,2024 observed candidates,49
4,Oxford St Ebbes retained,True
5,Inverness excluded,True


,region,sites
0,North West,9
1,South East,9
2,Yorkshire and The Humber,7
3,South West,6
4,West Midlands,6
5,North East,5
6,London,4
7,East Midlands,4
8,East of England,2


In [8]:
# Build graph samples
trainpm = daily[
    daily["site"].isin(sites)&daily["year"].between(START,END)
].pivot(index="date",columns="site",values="pm").reindex(columns=sites).sort_index()
validpm = daily[
    daily["site"].isin(sites)&daily["year"].eq(VALIDATION)
].pivot(index="date",columns="site",values="pm").reindex(columns=sites).sort_index()
nodes = candidates.set_index("site").reindex(sites).reset_index()
graphsamples = pd.DataFrame({
    "metric":[
        "candidates","training dates","validation dates","training observations",
        "validation observations","median training observations per site"
    ],
    "value":[
        len(sites),len(trainpm),len(validpm),trainpm.notna().sum().sum(),
        validpm.notna().sum().sum(),trainpm.notna().sum().median()
    ]
})
display(graphsamples)


,metric,value
0,candidates,52.0
1,training dates,765.0
2,validation dates,153.0
3,training observations,35940.0
4,validation observations,7123.0
5,median training observations per site,716.5


In [9]:
# Define graph tools
PRUNE = 1e-6
MIN_OVERLAP = 60

def graph(k,q,scale,return_graph=False):
    ids = list(sites)
    n = len(ids)
    k = min(int(k),n-1)
    coords = np.radians(nodes[["lat","lon"]].to_numpy(float))
    distance = 6371.0088*haversine_distances(coords)
    nearest = np.argsort(distance,axis=1)[:,1:k+1]
    mask = np.zeros((n,n),dtype=bool)
    mask[np.repeat(np.arange(n),k),nearest.reshape(-1)] = True
    mask = mask|mask.T
    np.fill_diagonal(mask,False)
    presence = trainpm.notna().astype(np.int32)
    overlap = (presence.T@presence).to_numpy()
    corr = trainpm.corr(min_periods=MIN_OVERLAP).fillna(0).to_numpy()
    corr = np.clip(corr,-1,1)
    np.fill_diagonal(corr,1)
    upper = np.triu(mask,1)
    sigma0 = float(np.median(distance[upper]))
    sigma = sigma0*float(scale)
    W = np.where(mask,np.exp(-(distance/sigma)**2)*np.maximum(corr,0)**float(q),0)
    W = (W+W.T)/2
    np.fill_diagonal(W,0)
    rawparts,_ = connected_components(csr_matrix(W>0),directed=False)
    cutoff = PRUNE*float(W.max())
    parts,_ = connected_components(csr_matrix(W>cutoff),directed=False)
    degree = W.sum(axis=1)
    L = np.diag(degree)-W
    values = np.maximum(np.linalg.eigvalsh(L),0)
    tol = np.finfo(float).eps*n*max(float(values[-1]),1)
    X = validpm.to_numpy(float)
    observed = np.isfinite(X)
    top = np.nan_to_num(X,nan=0)@W.T
    bottom = observed.astype(float)@W.T
    prediction = np.divide(top,bottom,out=np.full_like(top,np.nan),where=bottom>0)
    use = observed&np.isfinite(prediction)
    error = prediction[use]-X[use]
    result = {
        "k":k,
        "q":float(q),
        "scale":float(scale),
        "sigma0_km":sigma0,
        "sigma_km":sigma,
        "edges":int(np.count_nonzero(np.triu(W>0,1))),
        "raw_components":int(rawparts),
        "robust_components":int(parts),
        "numerical_zeros":int((values<=tol).sum()),
        "lambda1":float(values[1]),
        "minimum_degree":float(degree.min()),
        "validation_mae":float(np.mean(np.abs(error))),
        "validation_rmse":float(np.sqrt(np.mean(error**2))),
        "validation_coverage":float(use.sum()/observed.sum())
    }
    if not return_graph:
        return result
    values,vectors = np.linalg.eigh(L)
    network = {
        "ids":ids,"nodes":nodes.copy(),"distance":distance,"mask":mask,"overlap":overlap,
        "correlation":corr,"W":W,"degree":degree,"L":L,"values":np.maximum(values,0),
        "vectors":vectors,"prune":cutoff
    }
    return result,network


In [10]:
# Tune England graph
rows = []
for k,q,scale in product([8,10,12,15,20,25],[.5,1,2],[1,1.5,2,3,4,6,8]):
    if k<len(sites):
        rows.append(graph(k,q,scale))
tuning = pd.DataFrame(rows)
viable = tuning[
    tuning["raw_components"].eq(1)
    &tuning["robust_components"].eq(1)
    &tuning["numerical_zeros"].eq(1)
    &tuning["validation_coverage"].ge(.95)
].copy()
if viable.empty:
    raise RuntimeError("No robust England graph")
best = viable["validation_mae"].min()
near = viable[viable["validation_mae"]<=1.01*best]
graphchoice = near.sort_values(
    ["k","scale","validation_rmse","q"],
    ascending=[True,True,True,False]
).iloc[0]
graphresult,network = graph(
    int(graphchoice.k),float(graphchoice.q),float(graphchoice.scale),return_graph=True
)
baseline = tuning[
    tuning["k"].eq(10)&tuning["q"].eq(2)&tuning["scale"].eq(1)
].iloc[0]
graphaudit = pd.DataFrame([
    {
        "graph":"California fixed","k":baseline.k,"q":baseline.q,"scale":baseline.scale,
        "sigma_km":baseline.sigma_km,"components":baseline.robust_components,
        "lambda1":baseline.lambda1,"validation_mae":baseline.validation_mae
    },
    {
        "graph":"Selected","k":graphresult["k"],"q":graphresult["q"],"scale":graphresult["scale"],
        "sigma_km":graphresult["sigma_km"],"components":graphresult["robust_components"],
        "lambda1":graphresult["lambda1"],"validation_mae":graphresult["validation_mae"]
    }
])
display(tuning.sort_values("validation_mae").head(10).round(6))
display(graphaudit.round(6))


,k,q,scale,sigma0_km,sigma_km,edges,raw_components,robust_components,numerical_zeros,lambda1,minimum_degree,validation_mae,validation_rmse,validation_coverage
14,8,2.0,1.0,63.296565,63.296565,262,1,1,1,0.008657,0.028204,1.114521,1.684174,1.0
35,10,2.0,1.0,72.310047,72.310047,328,1,1,1,0.021076,0.089604,1.118969,1.681702,1.0
7,8,1.0,1.0,63.296565,63.296565,262,1,1,1,0.012214,0.034000,1.122411,1.691042,1.0
15,8,2.0,1.5,63.296565,94.944848,262,1,1,1,0.065845,0.437942,1.122767,1.679252,1.0
0,8,0.5,1.0,63.296565,63.296565,262,1,1,1,0.014499,0.037391,1.126771,1.695141,1.0
28,10,1.0,1.0,72.310047,72.310047,328,1,1,1,0.030081,0.109905,1.127504,1.690905,1.0
56,12,2.0,1.0,82.578885,82.578885,395,1,1,1,0.047597,0.225972,1.129182,1.690790,1.0
16,8,2.0,2.0,63.296565,126.593131,262,1,1,1,0.104003,1.114883,1.130927,1.686060,1.0
21,10,0.5,1.0,72.310047,72.310047,328,1,1,1,0.035923,0.121977,1.132384,1.696245,1.0
8,8,1.0,1.5,63.296565,94.944848,262,1,1,1,0.082872,0.544105,1.132642,1.690000,1.0


,graph,k,q,scale,sigma_km,components,lambda1,validation_mae
0,California fixed,10.0,2.0,1.0,72.310047,1.0,0.021076,1.118969
1,Selected,8.0,2.0,1.0,63.296565,1.0,0.008657,1.114521


In [11]:
# Load ONS geography
def arcgis(url,fields):
    rows = []
    start = 0
    while True:
        response = requests.get(
            url+"/query",
            params={
                "where":"1=1","outFields":",".join(fields),"returnGeometry":"false",
                "resultOffset":start,"resultRecordCount":1000,"f":"json"
            },
            timeout=120
        )
        response.raise_for_status()
        features = response.json().get("features",[])
        rows.extend(feature["attributes"] for feature in features)
        if len(features)<1000:
            break
        start += len(features)
    return pd.DataFrame(rows)

LOOKUP_URL = "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/MSOA21_BUA22_LAD22_RGN22_EW_LU_v2/FeatureServer/0"
MSOA_URL = "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3/FeatureServer/0"
lookup = arcgis(LOOKUP_URL,["MSOA21CD","MSOA21NM","RGN22CD","RGN22NM"])
centers = arcgis(MSOA_URL,["MSOA21CD","MSOA21NM","LONG","LAT"])
print("Lookup rows:",len(lookup))
print("Coordinate rows:",len(centers))
print("South East MSOAs:",lookup["RGN22NM"].eq(TARGET).sum())


Lookup rows: 7264
Coordinate rows: 7264
South East MSOAs: 1119


In [12]:
# Load Census population
NOMIS_URL = "https://www.nomisweb.co.uk/output/census/2021/census2021-ts001.zip"
response = requests.get(NOMIS_URL,timeout=180)
response.raise_for_status()
if not zipfile.is_zipfile(io.BytesIO(response.content)):
    raise ValueError("Invalid Nomis archive")
with zipfile.ZipFile(io.BytesIO(response.content)) as bundle:
    names = bundle.namelist()
    msoaname = [name for name in names if name.lower().endswith("census2021-ts001-msoa.csv")][0]
    regionname = [name for name in names if name.lower().endswith("census2021-ts001-rgn.csv")][0]
    census = pd.read_csv(bundle.open(msoaname),low_memory=False)
    regionpop = pd.read_csv(bundle.open(regionname),low_memory=False)

norm = lambda value:re.sub(r"[^a-z0-9]","",str(value).lower())
cols = {norm(column):column for column in census.columns}
codecol = cols["geographycode"]
popcol = cols["residencetypetotalmeasuresvalue"]
population = census[[codecol,popcol]].copy()
population.columns = ["MSOA21CD","population"]
population["population"] = pd.to_numeric(population["population"],errors="coerce")
rcols = {norm(column):column for column in regionpop.columns}
regiontotal = float(regionpop.loc[
    regionpop[rcols["geographycode"]].eq("E12000008"),
    rcols["residencetypetotalmeasuresvalue"]
].iloc[0])
print("MSOA population rows:",len(population))
print("Published South East population:",int(regiontotal))


MSOA population rows: 7264
Published South East population: 9278065


In [13]:
# Build South East target
target = lookup[lookup["RGN22NM"].eq(TARGET)][["MSOA21CD","MSOA21NM"]].merge(
    centers[["MSOA21CD","LONG","LAT"]],on="MSOA21CD",how="left",validate="one_to_one"
).merge(
    population,on="MSOA21CD",how="left",validate="one_to_one"
)
if target[["LONG","LAT","population"]].isna().any().any():
    raise ValueError("Incomplete South East census target")
distance = 6371.0088*haversine_distances(
    np.radians(target[["LAT","LONG"]].to_numpy(float)),
    np.radians(nodes[["lat","lon"]].to_numpy(float))
)
nearest = np.argmin(distance,axis=1)
target["site"] = np.asarray(sites)[nearest]
target["km"] = distance[np.arange(len(target)),nearest]
weights = target.groupby("site")["population"].sum().reindex(sites).fillna(0)
weights = weights/weights.sum()
support = int(weights.gt(0).sum())
weightsupport = float(1/np.sum(weights.to_numpy()**2))
targetaudit = pd.DataFrame({
    "metric":[
        "MSOAs","population used","published region population","relative population error",
        "positive-weight stations","effective weight support","median nearest km","p95 nearest km",
        "maximum nearest km","Oxford target weight"
    ],
    "value":[
        len(target),target["population"].sum(),regiontotal,
        abs(target["population"].sum()-regiontotal)/regiontotal,support,weightsupport,
        target["km"].median(),target["km"].quantile(.95),target["km"].max(),
        weights.get("UKA00518",0)
    ]
})
display(targetaudit)
display(weights[weights.gt(0)].sort_values(ascending=False).rename("weight").reset_index())


,metric,value
0,MSOAs,1.119000e+03
1,population used,9.278084e+06
2,published region population,9.278065e+06
3,relative population error,2.047841e-06
4,positive-weight stations,1.700000e+01
5,effective weight support,1.088425e+01
6,median nearest km,1.678515e+01
7,p95 nearest km,3.847895e+01
8,maximum nearest km,5.980309e+01
9,Oxford target weight,8.555172e-02


,site,weight
0,UKA00462,0.143606
1,UKA00553,0.135497
2,UKA00628,0.108622
3,UKA00572,0.095375
4,UKA00421,0.089598
5,UKA00518,0.085552
6,UKA00235,0.065180
7,UKA00472,0.063399
8,UKA00546,0.061410
9,UKA00251,0.045620


In [14]:
# Audit England spectrum
X = trainpm.to_numpy(float)
means = np.nanmean(X,axis=0)
filled = np.where(np.isnan(X),means[None,:],X)
coefficients = filled@network["vectors"]
total = np.sum(coefficients**2,axis=0)
centered = filled-filled.mean(axis=0,keepdims=True)
spatial = np.sum((centered@network["vectors"])**2,axis=0)
total = np.cumsum(total)/total.sum()
spatial = np.cumsum(spatial)/spatial.sum()
Ktotal = int(np.searchsorted(total,.95)+1)
K = int(np.searchsorted(spatial,.95)+1)
rows = []
for value in [10,15,20,25,30,35,40,45,50]:
    if value<len(sites):
        rows.append({
            "K":value,
            "total_energy":total[value-1],
            "centered_energy":spatial[value-1],
            "lambda_K":network["values"][value-1]
        })
spectral = pd.DataFrame(rows)
display(pd.DataFrame({
    "metric":["K95 total","K95 centered","candidate dimension"],
    "value":[Ktotal,K,len(sites)]
}))
display(spectral.round(6))


,metric,value
0,K95 total,15
1,K95 centered,38
2,candidate dimension,52


,K,total_energy,centered_energy,lambda_K
0,10,0.940256,0.779679,0.532157
1,15,0.954883,0.828232,1.656238
2,20,0.962757,0.858575,2.470646
3,25,0.968971,0.884663,3.144490
4,30,0.977359,0.915016,3.540445
5,35,0.983736,0.938476,3.804194
6,40,0.989429,0.959533,4.413152
7,45,0.995913,0.984572,4.756688
8,50,0.998417,0.993638,5.291836


In [15]:
# Build England target model
ids = list(network["ids"])
scaled = network["values"][:K]/network["values"][K-1]
basis = network["vectors"][:,:K]
decay = np.exp(-np.outer(np.array([0,.25,.5]),scaled))
blocks = basis[:,None,:]*decay[None,:,:]
g = np.exp(-scaled)*(basis.T@weights.reindex(ids).to_numpy(float))
info = np.einsum("vtk,vtl->vkl",blocks,blocks)
beta = .01*np.trace(info.sum(axis=0))/K
model = {
    "ids":ids,"K":K,"beta":beta,"blocks":blocks,"info":info,"g":g,
    "weights":weights.reindex(ids).to_numpy(float),"basis":basis,"graph":network
}
energy = np.sort(g**2)[::-1]
targetdim = int(np.searchsorted(np.cumsum(energy)/energy.sum(),.95)+1)
modelaudit = pd.DataFrame({
    "metric":["candidates","K","effective target dimension","beta","maximum ten-sensor rank"],
    "value":[len(ids),K,targetdim,beta,3*BUDGET]
})
display(modelaudit)


,metric,value
0,candidates,52.000000
1,K,38.000000
2,effective target dimension,10.000000
3,beta,0.024297
4,maximum ten-sensor rank,30.000000


In [16]:
# Define sensor design tools
def metrics(selected):
    if len(selected)==0:
        return {
            "span_error":1.,"mismatch":1.,"amplification":0.,"risk":1.,
            "rank":0,"sigma_min":0.,"full_state_risk":1.
        }
    B = blocks[selected].reshape(-1,K)
    normalizer = np.linalg.norm(g)
    exact = np.linalg.lstsq(B.T,g,rcond=None)[0]
    span = np.linalg.norm(B.T@exact-g)/normalizer
    M = beta*np.eye(K)+B.T@B
    h = np.linalg.solve(M,g)
    coefficients = B@h
    mismatch = np.linalg.norm(B.T@coefficients-g)/normalizer
    amplification = np.sqrt(beta)*np.linalg.norm(coefficients)/normalizer
    risk = np.sqrt(mismatch**2+amplification**2)
    rank = int(np.linalg.matrix_rank(B))
    singular = np.linalg.svd(B,compute_uv=False)
    sigma = float(singular[-1]) if rank==K else 0.
    return {
        "span_error":span,"mismatch":mismatch,"amplification":amplification,"risk":risk,
        "rank":rank,"sigma_min":sigma,"full_state_risk":np.sqrt(beta/(sigma**2+beta))
    }

def taps(budget):
    M = beta*np.eye(K)
    selected = []
    available = np.ones(len(ids),dtype=bool)
    normalizer = g@g
    for _ in range(budget):
        best = -1
        bestscore = np.inf
        for station in np.flatnonzero(available):
            score = beta*(g@np.linalg.solve(M+info[station],g))/normalizer
            if score<bestscore-1e-14:
                best = int(station)
                bestscore = score
        selected.append(best)
        available[best] = False
        M = M+info[best]
    return selected


In [17]:
# Scan England TAPS prefix
prefix = taps(len(ids))
rows = []
for budget in range(len(ids)+1):
    rows.append({"budget":budget,**metrics(prefix[:budget])})
curve = pd.DataFrame(rows)
exact = curve[curve["span_error"].lt(1e-6)]
full = curve[curve["rank"].ge(K)]
firstexact = int(exact["budget"].iloc[0]) if len(exact) else None
firstrank = int(full["budget"].iloc[0]) if len(full) else None
ten = curve[curve["budget"].eq(BUDGET)].iloc[0]
tapsaudit = pd.DataFrame({
    "metric":[
        "effective target dimension","first TAPS exact budget","first TAPS full-rank budget",
        "exact before full rank","ten-sensor span error","ten-sensor mismatch",
        "ten-sensor amplification","ten-sensor risk","ten-sensor rank","ten-sensor full-state risk"
    ],
    "value":[
        targetdim,firstexact,firstrank,
        firstexact is not None and (firstrank is None or firstexact<firstrank),
        ten["span_error"],ten["mismatch"],ten["amplification"],ten["risk"],ten["rank"],
        ten["full_state_risk"]
    ]
})
display(tapsaudit)
display(curve[curve["budget"].isin([0,3,5,7,10,13,15,20,25,30])].round(6))


,metric,value
0,effective target dimension,10
1,first TAPS exact budget,10
2,first TAPS full-rank budget,22
3,exact before full rank,True
4,ten-sensor span error,0.0
5,ten-sensor mismatch,0.158593
6,ten-sensor amplification,0.127684
7,ten-sensor risk,0.203605
8,ten-sensor rank,30.0
9,ten-sensor full-state risk,1.0


,budget,span_error,mismatch,amplification,risk,rank,sigma_min,full_state_risk
0,0,1.000000,1.000000,0.000000,1.000000,0,0.000000,1.0
3,3,0.077495,0.586144,0.205158,0.621011,9,0.000000,1.0
5,5,0.042973,0.417625,0.211932,0.468322,15,0.000000,1.0
7,7,0.000137,0.295052,0.183830,0.347634,21,0.000000,1.0
10,10,0.000000,0.158593,0.127684,0.203605,30,0.000000,1.0
13,13,0.000000,0.060480,0.115130,0.130049,33,0.000000,1.0
15,15,0.000000,0.036065,0.099243,0.105593,33,0.000000,1.0
20,20,0.000000,0.010444,0.091421,0.092016,36,0.000000,1.0
25,25,0.000000,0.008462,0.091315,0.091706,38,0.000000,1.0
30,30,0.000000,0.008458,0.091314,0.091705,38,0.000001,1.0


In [18]:
# Compare England placements
def raw(budget):
    M = beta*np.eye(K)
    selected = []
    available = np.ones(len(ids),dtype=bool)
    for _ in range(budget):
        h = np.linalg.solve(M,g)
        response = np.einsum("vtk,k->vt",blocks,h)
        score = np.sum(response**2,axis=1)
        score[~available] = -np.inf
        best = int(np.argmax(score))
        selected.append(best)
        available[best] = False
        M = M+info[best]
    return selected

targetorder = np.argsort(-model["weights"],kind="stable")[:BUDGET].tolist()
degreeorder = np.argsort(-network["degree"],kind="stable")[:BUDGET].tolist()
geographic = [int(np.argmin(network["distance"].max(axis=1)))]
while len(geographic)<BUDGET:
    nearest = network["distance"][:,geographic].min(axis=1)
    nearest[geographic] = -np.inf
    geographic.append(int(np.argmax(nearest)))

M = beta*np.eye(K)
doptimal = []
available = np.ones(len(ids),dtype=bool)
for _ in range(BUDGET):
    best = -1
    bestscore = -np.inf
    for station in np.flatnonzero(available):
        sign,score = np.linalg.slogdet(M+info[station])
        if sign>0 and score>bestscore+1e-12:
            best = int(station)
            bestscore = score
    doptimal.append(best)
    available[best] = False
    M = M+info[best]

_,_,pivots = qr(basis.T,pivoting=True,mode="economic")
placements = {
    "TAPS":prefix[:BUDGET],
    "Raw response":raw(BUDGET),
    "Target weight":targetorder,
    "Weighted degree":degreeorder,
    "Geographic coverage":geographic,
    "D-optimal":doptimal,
    "QR pivoting":[int(value) for value in pivots[:BUDGET]]
}
rows = []
for method,chosen in placements.items():
    rows.append({
        "method":method,
        "overlap_with_taps":len(set(chosen)&set(prefix[:BUDGET])),
        "sites":"|".join(np.asarray(ids)[chosen]),
        **metrics(chosen)
    })
comparison = pd.DataFrame(rows).sort_values("risk")
display(comparison[
    ["method","span_error","mismatch","amplification","risk","rank","full_state_risk","overlap_with_taps"]
].round(6))
print("TAPS sites:","|".join(np.asarray(ids)[placements["TAPS"]]))


,method,span_error,mismatch,amplification,risk,rank,full_state_risk,overlap_with_taps
1,Raw response,0.000000,0.140561,0.119190,0.184292,30,1.0,9
0,TAPS,0.000000,0.158593,0.127684,0.203605,30,1.0,10
2,Target weight,0.000000,0.187102,0.133728,0.229979,30,1.0,9
3,Weighted degree,0.023615,0.664128,0.323951,0.738925,30,1.0,1
4,Geographic coverage,0.051905,0.703827,0.322558,0.774219,30,1.0,2
5,D-optimal,0.149227,0.877472,0.288311,0.923624,30,1.0,0
6,QR pivoting,0.265891,0.998726,0.033479,0.999287,30,1.0,0


TAPS sites: UKA00472|UKA00628|UKA00553|UKA00462|UKA00421|UKA00518|UKA00546|UKA00235|UKA00251|UKA00614


In [19]:
# Build forecast samples
COVERAGE = .85
matrix = daily[daily["site"].isin(sites)].pivot(
    index="date",columns="site",values="pm"
).reindex(columns=ids).sort_index()

def targetvalue(matrix,weights,threshold):
    weights = weights[weights>0].copy()
    block = matrix.reindex(columns=weights.index)
    coverage = block.notna().mul(weights,axis=1).sum(axis=1)
    value = block.mul(weights,axis=1).sum(axis=1,min_count=1)/coverage.where(coverage>0)
    return value.where(coverage>=threshold),coverage

truth,truthcoverage = targetvalue(matrix,weights.reindex(ids),COVERAGE)
available = set(matrix.index)
rows = []
for date in matrix.index:
    dates = [date-pd.Timedelta(days=4),date-pd.Timedelta(days=3),date-pd.Timedelta(days=2)]
    if all(value in available and value.year==date.year for value in dates):
        block = matrix.loc[dates]
        rows.append([
            date,*dates,int(date.year),float(block.notna().mean().mean()),
            int(block.notna().all(axis=0).sum())
        ])
samples = pd.DataFrame(
    rows,
    columns=[
        "target_date","first_date","middle_date","latest_date","year",
        "input_fraction","complete_stations"
    ]
)
samples["truth"] = truth.reindex(samples["target_date"]).to_numpy(float)
samples["coverage"] = truthcoverage.reindex(samples["target_date"]).to_numpy(float)
rows = []
for year in range(START,TEST+1):
    part = samples[samples["year"].eq(year)&samples["truth"].notna()]
    rows.append({
        "year":year,
        "truth_days":len(part),
        "median_coverage":part["coverage"].median() if len(part) else np.nan,
        "median_input_fraction":part["input_fraction"].median() if len(part) else np.nan,
        "minimum_complete_stations":int(part["complete_stations"].min()) if len(part) else 0
    })
forecastaudit = pd.DataFrame(rows)
display(forecastaudit.round(4))


,year,truth_days,median_coverage,median_input_fraction,minimum_complete_stations
0,2018,84,0.9104,0.9103,32
1,2019,74,0.8564,0.9327,38
2,2020,106,0.9052,0.9231,39
3,2021,130,0.9849,0.9167,38
4,2022,126,0.9849,0.9359,40
5,2023,48,0.8914,0.9038,42
6,2024,100,0.8895,0.9135,42


In [20]:
# Define forecast tools
def score(prediction,truth):
    prediction = np.asarray(prediction,float)
    truth = np.asarray(truth,float)
    use = np.isfinite(prediction)&np.isfinite(truth)
    prediction = prediction[use]
    truth = truth[use]
    if not len(truth):
        return {"n":0,"mae":np.nan,"rmse":np.nan,"bias":np.nan,"correlation":np.nan}
    error = prediction-truth
    correlation = (
        np.corrcoef(prediction,truth)[0,1]
        if len(truth)>1 and np.std(prediction)>0 and np.std(truth)>0
        else np.nan
    )
    return {
        "n":len(truth),
        "mae":float(np.mean(np.abs(error))),
        "rmse":float(np.sqrt(np.mean(error**2))),
        "bias":float(np.mean(error)),
        "correlation":float(correlation) if np.isfinite(correlation) else np.nan
    }

def calibrate(prediction,truth):
    prediction = np.asarray(prediction,float)
    truth = np.asarray(truth,float)
    use = np.isfinite(prediction)&np.isfinite(truth)
    X = np.column_stack([np.ones(use.sum()),prediction[use]])
    return np.linalg.lstsq(X,truth[use],rcond=None)[0]

def features(frame,selected):
    use = [ids[index] for index in selected]
    rows = []
    for row in frame.itertuples():
        block = matrix.loc[
            [row.first_date,row.middle_date,row.latest_date],use
        ].to_numpy(float).T.reshape(-1)
        angle = 2*np.pi*row.target_date.dayofyear/365.25
        rows.append(np.r_[block,np.sin(angle),np.cos(angle)])
    return np.asarray(rows,float)

def latest(frame,selected):
    use = [ids[index] for index in selected]
    return np.asarray([
        matrix.loc[row.latest_date,use].mean() for row in frame.itertuples()
    ],float)

def graphpred(frame,selected):
    use = [ids[index] for index in selected]
    design = blocks[selected].reshape(-1,K)
    predictions = []
    counts = []
    for row in frame.itertuples():
        y = matrix.loc[
            [row.first_date,row.middle_date,row.latest_date],use
        ].to_numpy(float).T.reshape(-1)
        observed = np.isfinite(y)
        A = design[observed]
        M = beta*np.eye(K)+A.T@A
        state = np.linalg.solve(M,A.T@y[observed])
        predictions.append(g@state)
        counts.append(int(observed.sum()))
    return np.asarray(predictions),np.asarray(counts)


In [21]:
# Tune England predictor
trainset = samples[samples["year"].between(START,END)&samples["truth"].notna()]
validset = samples[samples["year"].eq(VALIDATION)&samples["truth"].notna()]
ytrain = trainset["truth"].to_numpy(float)
yvalid = validset["truth"].to_numpy(float)
tapssites = placements["TAPS"]
Xtrain = features(trainset,tapssites)
Xvalid = features(validset,tapssites)
rows = []
for depth,leaf,feature in product([4,8,None],[2,5],[.5,1.0]):
    forest = Pipeline([
        ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
        ("model",RandomForestRegressor(
            n_estimators=200,max_depth=depth,min_samples_leaf=leaf,
            max_features=feature,random_state=SEED,n_jobs=-1
        ))
    ])
    forest.fit(Xtrain,ytrain)
    rows.append({
        "depth":depth,
        "leaf":leaf,
        "features":feature,
        "mae":score(forest.predict(Xvalid),yvalid)["mae"]
    })
rftrials = pd.DataFrame(rows).sort_values("mae")
rfchoice = rftrials.iloc[0]
display(rftrials.round(6))
print("Selected depth:",rfchoice["depth"])
print("Selected leaf:",rfchoice["leaf"])
print("Selected features:",rfchoice["features"])


,depth,leaf,features,mae
11,NaN,5,1.0,2.437440
7,8.0,5,1.0,2.444927
10,NaN,5,0.5,2.488900
3,4.0,5,1.0,2.500574
6,8.0,5,0.5,2.514748
2,4.0,5,0.5,2.547421
9,NaN,2,1.0,2.610311
8,NaN,2,0.5,2.624137
5,8.0,2,1.0,2.631252
4,8.0,2,0.5,2.635348


Selected depth: nan
Selected leaf: 5.0
Selected features: 1.0


In [22]:
# Evaluate England validation
depth = None if pd.isna(rfchoice["depth"]) else int(rfchoice["depth"])
leaf = int(rfchoice["leaf"])
feature = float(rfchoice["features"])
predictions = {"Training mean":np.full(len(yvalid),ytrain.mean())}
gtrain,_ = graphpred(trainset,tapssites)
gvalid,_ = graphpred(validset,tapssites)
intercept,slope = calibrate(gtrain,ytrain)
predictions["Graph diffusion"] = intercept+slope*gvalid
ptrain = latest(trainset,tapssites)
pvalid = latest(validset,tapssites)
fill = np.nanmedian(ptrain)
intercept,slope = calibrate(np.nan_to_num(ptrain,nan=fill),ytrain)
predictions["Persistence"] = intercept+slope*np.nan_to_num(pvalid,nan=fill)
forest = Pipeline([
    ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
    ("model",RandomForestRegressor(
        n_estimators=500,max_depth=depth,min_samples_leaf=leaf,
        max_features=feature,random_state=SEED,n_jobs=-1
    ))
])
forest.fit(Xtrain,ytrain)
predictions["Random forest"] = forest.predict(Xvalid)
validation = pd.DataFrame([
    {"model":name,**score(prediction,yvalid)} for name,prediction in predictions.items()
]).sort_values("mae")
display(validation.round(6))


,model,n,mae,rmse,bias,correlation
3,Random forest,48,2.426309,3.219004,-0.179579,0.612987
1,Graph diffusion,48,2.559593,3.498307,-0.430477,0.539334
2,Persistence,48,2.608372,3.456548,-0.287824,0.553827
0,Training mean,48,2.977760,4.074325,-0.564711,-0.000000


In [23]:
# Evaluate England test
development = samples[samples["year"].le(VALIDATION)&samples["truth"].notna()].copy()
test = samples[samples["year"].eq(TEST)&samples["truth"].notna()].copy()
ydev = development["truth"].to_numpy(float)
ytest = test["truth"].to_numpy(float)
Xdev = features(development,tapssites)
Xtest = features(test,tapssites)
forest = Pipeline([
    ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
    ("model",RandomForestRegressor(
        n_estimators=500,max_depth=depth,min_samples_leaf=leaf,
        max_features=feature,random_state=SEED,n_jobs=-1
    ))
])
forest.fit(Xdev,ydev)
predictions = {
    "Training mean":np.full(len(ytest),ydev.mean()),
    "Random forest":forest.predict(Xtest)
}
gdev,_ = graphpred(development,tapssites)
gtest,gobserved = graphpred(test,tapssites)
intercept,slope = calibrate(gdev,ydev)
predictions["Graph diffusion"] = intercept+slope*gtest
pdev = latest(development,tapssites)
ptest = latest(test,tapssites)
fill = np.nanmedian(pdev)
intercept,slope = calibrate(np.nan_to_num(pdev,nan=fill),ydev)
predictions["Persistence"] = intercept+slope*np.nan_to_num(ptest,nan=fill)
testresults = pd.DataFrame([
    {"model":name,**score(prediction,ytest)} for name,prediction in predictions.items()
]).sort_values("mae")
testpred = pd.DataFrame({
    "target_date":test["target_date"].to_numpy(),
    "truth":ytest,
    "graph_observed":gobserved
})
for name,prediction in predictions.items():
    testpred[name] = prediction
display(testresults.round(6))
display(pd.DataFrame({
    "metric":["test days","minimum graph observations","median graph observations"],
    "value":[len(test),int(gobserved.min()),float(np.median(gobserved))]
}))


,model,n,mae,rmse,bias,correlation
1,Random forest,100,2.303620,3.289683,0.836000,0.264045
3,Persistence,100,2.325406,3.245261,0.953403,0.318570
2,Graph diffusion,100,2.428470,3.364360,1.008532,0.225583
0,Training mean,100,2.797076,3.579945,1.464059,NaN


,metric,value
0,test days,100.0
1,minimum graph observations,27.0
2,median graph observations,30.0


In [24]:
# Evaluate held-out placements
rows = []
predrows = []
for method,chosen in placements.items():
    gdev,_ = graphpred(development,chosen)
    gtest,_ = graphpred(test,chosen)
    intercept,slope = calibrate(gdev,ydev)
    graphprediction = intercept+slope*gtest
    Xdev = features(development,chosen)
    Xtest = features(test,chosen)
    forest = Pipeline([
        ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
        ("model",RandomForestRegressor(
            n_estimators=500,max_depth=depth,min_samples_leaf=leaf,
            max_features=feature,random_state=SEED,n_jobs=-1
        ))
    ])
    forest.fit(Xdev,ydev)
    forestprediction = forest.predict(Xtest)
    for predictor,prediction in [
        ("Graph diffusion",graphprediction),
        ("Random forest",forestprediction)
    ]:
        rows.append({"placement":method,"predictor":predictor,**score(prediction,ytest)})
        predrows.extend({
            "target_date":date,"truth":truth,"placement":method,
            "predictor":predictor,"prediction":value
        } for date,truth,value in zip(test["target_date"],ytest,prediction))
heldout = pd.DataFrame(rows).sort_values(["predictor","mae"])
heldoutpred = pd.DataFrame(predrows)
if len(heldout)!=14:
    raise ValueError("Incomplete placement evaluation")
display(heldout.round(6))


,placement,predictor,n,mae,rmse,bias,correlation
2,Raw response,Graph diffusion,100,2.174295,3.202614,0.694582,0.302705
4,Target weight,Graph diffusion,100,2.411800,3.338138,0.993762,0.245352
0,TAPS,Graph diffusion,100,2.428470,3.364360,1.008532,0.225583
6,Weighted degree,Graph diffusion,100,2.497079,3.339810,1.154578,0.282980
8,Geographic coverage,Graph diffusion,100,2.586756,3.365169,1.317721,0.318783
10,D-optimal,Graph diffusion,100,2.632635,3.425590,1.348744,0.266515
12,QR pivoting,Graph diffusion,100,2.774122,3.538033,1.408427,0.155307
11,D-optimal,Random forest,100,2.232889,3.242184,0.763553,0.287971
3,Raw response,Random forest,100,2.247852,3.263398,0.677387,0.258355
1,TAPS,Random forest,100,2.303620,3.289683,0.836000,0.264045


In [25]:
# Benchmark random placements
rng = np.random.default_rng(SEED)
seen = set()
while len(seen)<RANDOM:
    seen.add(tuple(sorted(rng.choice(len(ids),BUDGET,replace=False).tolist())))
rows = []
for draw,chosen in enumerate(sorted(seen)):
    chosen = list(chosen)
    gdev,_ = graphpred(development,chosen)
    gtest,_ = graphpred(test,chosen)
    intercept,slope = calibrate(gdev,ydev)
    prediction = intercept+slope*gtest
    rows.append({
        "draw":draw,
        "sites":"|".join(np.asarray(ids)[chosen]),
        **score(prediction,ytest)
    })
random = pd.DataFrame(rows)
tapsmae = float(heldout[
    heldout["placement"].eq("TAPS")&heldout["predictor"].eq("Graph diffusion")
]["mae"].iloc[0])
randomaudit = pd.DataFrame({
    "metric":[
        "random sets","unique sets","TAPS graph MAE","random mean MAE","random median MAE",
        "random p05 MAE","random p95 MAE","TAPS better than random fraction"
    ],
    "value":[
        len(random),random["sites"].nunique(),tapsmae,random["mae"].mean(),
        random["mae"].median(),random["mae"].quantile(.05),random["mae"].quantile(.95),
        random["mae"].gt(tapsmae).mean()
    ]
})
display(randomaudit.round(6))


,metric,value
0,random sets,200.000000
1,unique sets,200.000000
2,TAPS graph MAE,2.428470
3,random mean MAE,2.465808
4,random median MAE,2.522592
5,random p05 MAE,2.050967
6,random p95 MAE,2.720223
7,TAPS better than random fraction,0.690000


In [26]:
# Bootstrap England comparisons
def boot(frame,a,b,reps=BOOTSTRAP,seed=SEED):
    data = frame.sort_values("target_date").reset_index(drop=True).copy()
    data["target_date"] = pd.to_datetime(data["target_date"])
    data["block"] = ((data["target_date"].dt.dayofyear-1)//7).astype(int)
    truth = data["truth"].to_numpy(float)
    pa = data[a].to_numpy(float)
    pb = data[b].to_numpy(float)
    groups = {
        block:data.index[data["block"].eq(block)].to_numpy()
        for block in data["block"].unique()
    }
    keys = np.array(list(groups))
    rng = np.random.default_rng(seed)
    maediff = np.empty(reps)
    rmsediff = np.empty(reps)
    for rep in range(reps):
        index = np.concatenate([
            groups[block] for block in rng.choice(keys,len(keys),replace=True)
        ])
        ea = pa[index]-truth[index]
        eb = pb[index]-truth[index]
        maediff[rep] = np.mean(np.abs(ea))-np.mean(np.abs(eb))
        rmsediff[rep] = np.sqrt(np.mean(ea**2))-np.sqrt(np.mean(eb**2))
    return {
        "mae_diff":np.mean(np.abs(pa-truth))-np.mean(np.abs(pb-truth)),
        "mae_low":np.quantile(maediff,.025),
        "mae_high":np.quantile(maediff,.975),
        "rmse_diff":np.sqrt(np.mean((pa-truth)**2))-np.sqrt(np.mean((pb-truth)**2)),
        "rmse_low":np.quantile(rmsediff,.025),
        "rmse_high":np.quantile(rmsediff,.975)
    }

rows = []
for i,(a,b) in enumerate([
    ("Random forest","Graph diffusion"),
    ("Graph diffusion","Persistence"),
    ("Random forest","Persistence")
]):
    rows.append({"model_a":a,"model_b":b,**boot(testpred,a,b,seed=SEED+i)})
modelboot = pd.DataFrame(rows)

rows = []
for i,predictor in enumerate(["Graph diffusion","Random forest"]):
    block = heldoutpred[
        heldoutpred["predictor"].eq(predictor)
        &heldoutpred["placement"].isin(["TAPS","Raw response","Target weight"])
    ]
    wide = block.pivot(
        index=["target_date","truth"],columns="placement",values="prediction"
    ).reset_index()
    for j,(a,b) in enumerate([("TAPS","Raw response"),("TAPS","Target weight")]):
        rows.append({
            "predictor":predictor,"placement_a":a,"placement_b":b,
            **boot(wide,a,b,seed=100+10*i+j)
        })
placementboot = pd.DataFrame(rows)
display(modelboot.round(6))
display(placementboot.round(6))


,model_a,model_b,mae_diff,mae_low,mae_high,rmse_diff,rmse_low,rmse_high
0,Random forest,Graph diffusion,-0.124850,-0.346109,0.098052,-0.074677,-0.293791,0.137076
1,Graph diffusion,Persistence,0.103064,0.011351,0.213946,0.119099,-0.019940,0.240026
2,Random forest,Persistence,-0.021786,-0.200589,0.155142,0.044421,-0.134808,0.170184


,predictor,placement_a,placement_b,mae_diff,mae_low,mae_high,rmse_diff,rmse_low,rmse_high
0,Graph diffusion,TAPS,Raw response,0.254175,0.114276,0.456732,0.161746,0.030656,0.406731
1,Graph diffusion,TAPS,Target weight,0.016670,-0.015152,0.056822,0.026222,-0.021550,0.072731
2,Random forest,TAPS,Raw response,0.055769,0.000349,0.112154,0.026285,-0.019404,0.097862
3,Random forest,TAPS,Target weight,-0.145989,-0.297318,-0.017825,-0.066263,-0.197541,0.035652


In [27]:
# Benchmark model random sets
rows = []
for draw,chosen in enumerate(sorted(seen)):
    rows.append({"draw":draw,**metrics(list(chosen))})
modelrandom = pd.DataFrame(rows)
tapsrisk = metrics(placements["TAPS"])["risk"]
rawrisk = metrics(placements["Raw response"])["risk"]
modelrandomaudit = pd.DataFrame({
    "metric":[
        "random sets","random mean risk","random median risk","random p05 risk","random p95 risk",
        "TAPS risk","TAPS better than random fraction","Raw-response risk",
        "Raw response better than random fraction"
    ],
    "value":[
        len(modelrandom),modelrandom["risk"].mean(),modelrandom["risk"].median(),
        modelrandom["risk"].quantile(.05),modelrandom["risk"].quantile(.95),tapsrisk,
        modelrandom["risk"].gt(tapsrisk).mean(),rawrisk,modelrandom["risk"].gt(rawrisk).mean()
    ]
})
display(modelrandomaudit.round(6))


,metric,value
0,random sets,200.000000
1,random mean risk,0.758698
2,random median risk,0.750288
3,random p05 risk,0.621322
4,random p95 risk,0.932492
5,TAPS risk,0.203605
6,TAPS better than random fraction,1.000000
7,Raw-response risk,0.184292
8,Raw response better than random fraction,1.000000


In [28]:
# Rank held-out placements
rows = []
for method in placements:
    mae = float(heldout[
        heldout["placement"].eq(method)&heldout["predictor"].eq("Graph diffusion")
    ]["mae"].iloc[0])
    rows.append({
        "placement":method,
        "mae":mae,
        "random_mean":random["mae"].mean(),
        "random_median":random["mae"].median(),
        "percent_random_beaten":100*random["mae"].gt(mae).mean()
    })
randomrank = pd.DataFrame(rows).sort_values("mae")
display(randomrank.round(4))


,placement,mae,random_mean,random_median,percent_random_beaten
1,Raw response,2.1743,2.4658,2.5226,89.0
2,Target weight,2.4118,2.4658,2.5226,69.5
0,TAPS,2.4285,2.4658,2.5226,69.0
3,Weighted degree,2.4971,2.4658,2.5226,59.0
4,Geographic coverage,2.5868,2.4658,2.5226,30.5
5,D-optimal,2.6326,2.4658,2.5226,15.0
6,QR pivoting,2.7741,2.4658,2.5226,2.0


In [29]:
# Bootstrap placement leaders
def compare(predictor,a,b,seed):
    part = heldoutpred[
        heldoutpred["predictor"].eq(predictor)&heldoutpred["placement"].isin([a,b])
    ]
    wide = part.pivot(
        index=["target_date","truth"],columns="placement",values="prediction"
    ).reset_index()
    return {
        "predictor":predictor,"placement_a":a,"placement_b":b,
        **boot(wide,a,b,seed=seed)
    }

leaders = pd.DataFrame([
    compare("Graph diffusion","Raw response","TAPS",201),
    compare("Graph diffusion","Raw response","Target weight",202),
    compare("Random forest","D-optimal","TAPS",203),
    compare("Random forest","D-optimal","Raw response",204),
    compare("Random forest","Raw response","TAPS",205)
])
display(leaders.round(6))


,predictor,placement_a,placement_b,mae_diff,mae_low,mae_high,rmse_diff,rmse_low,rmse_high
0,Graph diffusion,Raw response,TAPS,-0.254175,-0.460241,-0.094597,-0.161746,-0.426587,-0.014359
1,Graph diffusion,Raw response,Target weight,-0.237505,-0.397288,-0.110890,-0.135524,-0.339497,-0.022350
2,Random forest,D-optimal,TAPS,-0.070731,-0.272237,0.098237,-0.047499,-0.291293,0.125506
3,Random forest,D-optimal,Raw response,-0.014962,-0.214157,0.159070,-0.021215,-0.234778,0.156897
4,Random forest,Raw response,TAPS,-0.055769,-0.109917,-0.001435,-0.026285,-0.099304,0.016131


In [30]:
# Freeze England summary
bestmodel = testresults.sort_values("mae").iloc[0]
bestgraph = heldout[heldout["predictor"].eq("Graph diffusion")].sort_values("mae").iloc[0]
bestrf = heldout[heldout["predictor"].eq("Random forest")].sort_values("mae").iloc[0]
summary = pd.DataFrame([
    ["candidates",len(ids)],
    ["target","South East England"],
    ["target_population",int(target["population"].sum())],
    ["published_population",int(regiontotal)],
    ["target_support",int(weights.gt(0).sum())],
    ["target_effective_support",weightsupport],
    ["Oxford_target_weight",float(weights.get("UKA00518",0))],
    ["K",K],
    ["effective_target_dimension",targetdim],
    ["first_exact_budget",firstexact],
    ["first_full_rank_budget",firstrank],
    ["taps_model_risk_10",tapsrisk],
    ["raw_model_risk_10",rawrisk],
    ["taps_model_random_fraction",float(modelrandom["risk"].gt(tapsrisk).mean())],
    ["raw_model_random_fraction",float(modelrandom["risk"].gt(rawrisk).mean())],
    ["best_2024_model",bestmodel["model"]],
    ["taps_rf_mae",float(testresults.loc[testresults["model"].eq("Random forest"),"mae"].iloc[0])],
    ["best_graph_placement",bestgraph["placement"]],
    ["best_graph_mae",float(bestgraph["mae"])],
    ["best_rf_placement",bestrf["placement"]],
    ["best_rf_mae",float(bestrf["mae"])],
    ["taps_graph_random_fraction",float(random["mae"].gt(tapsmae).mean())],
    ["test_days",len(test)]
],columns=["result","value"])
display(summary)


,result,value
0,candidates,52
1,target,South East England
2,target_population,9278084
3,published_population,9278065
4,target_support,17
5,target_effective_support,10.884252
6,Oxford_target_weight,0.085552
7,K,38
8,effective_target_dimension,10
9,first_exact_budget,10


In [31]:
# Build England exports
OUT.mkdir(exist_ok=True)
weighttable = pd.DataFrame({"site":ids,"weight":weights.reindex(ids).to_numpy(float)})
graphedges = []
W = network["W"]
for i in range(len(ids)):
    for j in range(i+1,len(ids)):
        if W[i,j]>0:
            graphedges.append({
                "site_a":ids[i],"site_b":ids[j],"weight":W[i,j],
                "distance_km":network["distance"][i,j],
                "correlation":network["correlation"][i,j],
                "overlap":network["overlap"][i,j]
            })
graphedges = pd.DataFrame(graphedges)
selection = pd.DataFrame([
    {"method":method,"sites":"|".join(np.asarray(ids)[chosen])}
    for method,chosen in placements.items()
])
source = pd.DataFrame([
    [
        "Defra UK-AIR AURN metadata",
        f"{AURN}/networks/find-sites?action=results&country_id=9999&group_id=4&location_type=9999&pollutant=&region_id=9999&site_name=&view=advanced",
        "current AURN station index",
        ACCESS
    ],
    [
        "Defra UK-AIR AURN closed sites",
        f"{AURN}/networks/aurn-sites",
        "historical AURN station index",
        ACCESS
    ],
    [
        "Defra UK-AIR PM2.5 files",
        f"{AURN}/datastore/data_files/site_pol_data/{{site}}_PM25_{{year}}.csv",
        "2018-2024 hourly PM2.5",
        ACCESS
    ],
    [
        "Nomis Census 2021 TS001",
        NOMIS_URL,
        "2021 usual-resident population",
        ACCESS
    ],
    [
        "ONS MSOA-region lookup",
        LOOKUP_URL,
        "2021 MSOA to English region geography",
        ACCESS
    ],
    [
        "ONS MSOA geography",
        MSOA_URL,
        "2021 MSOA coordinates",
        ACCESS
    ],
    [
        "IQAir England wildfire map",
        IQAIR,
        "motivating visualization only; not analytical input",
        ACCESS
    ]
],columns=["source","url","use","access_date"])


In [32]:
# Write England tables
tables = {
    "protocol.csv":protocol,
    "source_manifest.csv":source,
    "aurn_metadata.csv":meta,
    "candidate_coverage.csv":stats,
    "candidate_audit.csv":candidateaudit,
    "candidates.csv":candidates,
    "graph_samples.csv":graphsamples,
    "graph_tuning.csv":tuning,
    "graph_audit.csv":graphaudit,
    "graph_edges.csv":graphedges,
    "target_msoas.csv":target,
    "target_weights.csv":weighttable,
    "target_audit.csv":targetaudit,
    "spectral_audit.csv":spectral,
    "model_audit.csv":modelaudit,
    "taps_audit.csv":tapsaudit,
    "taps_curve.csv":curve,
    "model_placements.csv":comparison,
    "selected_sites.csv":selection,
    "model_random.csv":modelrandom,
    "model_random_audit.csv":modelrandomaudit,
    "forecast_audit.csv":forecastaudit,
    "rf_tuning.csv":rftrials,
    "validation_results.csv":validation,
    "test_results.csv":testresults,
    "heldout_placements.csv":heldout,
    "heldout_predictions.csv":heldoutpred,
    "heldout_random.csv":random,
    "heldout_random_audit.csv":randomaudit,
    "heldout_random_rank.csv":randomrank,
    "model_bootstrap.csv":modelboot,
    "placement_bootstrap.csv":placementboot,
    "placement_leaders_bootstrap.csv":leaders,
    "final_summary.csv":summary
}
for name,frame in tables.items():
    frame.to_csv(OUT/name,index=False)
print("Tables:",len(tables))


Tables: 34


In [35]:
# Archive England results
from google.colab import files

graphrow = graphaudit[
    graphaudit["graph"].eq("Selected")
].iloc[0]

manifest = {
    "experiment":"England external replication",
    "target":"South East England population-weighted PM2.5",
    "qualification":"2018-2022",
    "validation":2023,
    "test":2024,
    "season":"June-October",
    "candidates":len(ids),
    "graph":{
        "k":int(graphrow["k"]),
        "q":float(graphrow["q"]),
        "scale":float(graphrow["scale"]),
        "K":int(K)
    },
    "sensor_budget":BUDGET,
    "random_sets":RANDOM,
    "bootstrap_replicates":BOOTSTRAP,
    "seed":SEED,
    "target_coverage_threshold":COVERAGE,
    "target_effective_support":weightsupport,
    "effective_target_dimension":targetdim,
    "test_days":len(test),
    "first_exact_taps_prefix":firstexact,
    "first_full_rank_taps_prefix":firstrank,
    "greedy_not_global_optimum":True,
    "access_date":ACCESS
}

with open(OUT/"manifest.json","w") as file:
    json.dump(manifest,file,indent=2)

archive = Path("england.zip")

with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT.iterdir()):
        bundle.write(path,arcname=path.name)

with zipfile.ZipFile(archive) as bundle:
    bad = bundle.testzip()
    members = bundle.namelist()

print("Archive:",archive)
print("Size MB:",round(archive.stat().st_size/1e6,3))
print("Members:",len(members))
print("Corrupt member:",bad)
print("Graph:",manifest["graph"])
print("Target effective support:",manifest["target_effective_support"])
print("Effective target dimension:",manifest["effective_target_dimension"])
print("Test days:",manifest["test_days"])

if bad is not None:
    raise ValueError("Corrupt ZIP member: "+bad)

files.download(str(archive))

Archive: england.zip
Size MB: 0.107
Members: 35
Corrupt member: None
Graph: {'k': 8, 'q': 2.0, 'scale': 1.0, 'K': 38}
Target effective support: 10.884251592215179
Effective target dimension: 10
Test days: 100


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>